In [1]:
from run_stacking import *

2025-12-16 10:20:43.986604: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
dataset = 'Beef'

X_train, y_train, X_test, y_test = utils.load_dataset(dataset)
print(X_train.shape, y_train.shape)

(30, 1, 470) (30,)


In [3]:
class StackerV4(BaseClassifier):
    def __init__(self, random_state=None, n_repetitions = 1, k_folds=10):
        super().__init__()
        self.n_repetitions = n_repetitions
        self.k_folds = k_folds
        self.random_state = random_state
        self.cv_splits = None

    def get_model(self, name):
        if name == 'multirocket-ridgecv':
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        'hydra_': SparseScaler(),
                        'multirocket_': StandardScaler()
                    },
                    verbose=False
                ),
                RidgeClassifierCVIndicator(alphas=np.logspace(-3, 3, 10))
            )
            return pipe
        elif name == 'rstsf':
            return RSTSF(random_state=self.random_state, n_jobs=-1, n_estimators=100)
        elif name == 'quant-etc':
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        'quant_': NoScaler(),
                    },
                    verbose=False
                ),
                ExtraTreesClassifier(
                    n_estimators=200,
                    max_features=0.1,
                    criterion="entropy",
                    random_state=self.random_state, # pass this in 
                    n_jobs=-1
                )
            )
            return pipe
        elif name == 'rdst-ridgecv':
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        'rdst_': StandardScaler(),
                    },
                    verbose=False
                ),
                RidgeClassifierCVIndicator(alphas=np.logspace(-4, 4, 20))
            )
            return pipe
        elif name == 'rdst-robustscale-ridgecv':
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        'rdst_': RobustScaler(),
                    },
                    verbose=False
                ),
                RidgeClassifierCVIndicator(alphas=np.logspace(-4, 4, 20))
            )
            return pipe
        elif name == 'catch22-quant-et':
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        'catch22_': NoScaler(),
                        'quant_': NoScaler(),
                    },
                    verbose=False
                ),
                ExtraTreesClassifier(
                    n_estimators=200,
                    max_features=0.1,
                    criterion="entropy",
                    random_state=self.random_state, # pass this in 
                    n_jobs=-1
                )
            )
            return pipe
        elif name == 'probability-linear-svc':
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        'proba_': StandardScaler(),
                    },
                    verbose=False
                ),
                SVC(kernel='linear', probability=True, random_state=self.random_state)
            )
            return pipe
        elif name == 'probability-et':
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        'proba_': StandardScaler(),
                    },
                    verbose=False
                ),
                ExtraTreesClassifier(
                    n_estimators=500,
                    #max_features=0.3,
                    #criterion="entropy",
                    random_state=self.random_state, # pass this in 
                    n_jobs=-1,
                    bootstrap=True
                )
            )
            return pipe
        elif name == 'probability-ridgecv':
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        'proba_': StandardScaler(),
                    },
                    verbose=False
                ),
                RidgeClassifierCVIndicator(alphas=np.logspace(-3, 3, 10))
            )
            return pipe
        elif name == 'probability-rf':
            pipe = make_pipeline(
                MultiScaler(
                    scalers={
                        'proba_': StandardScaler(),
                    },
                    verbose=False
                ),
                RandomForestClassifier(n_estimators=200, random_state=self.random_state, n_jobs=-1)
            )
            return pipe
        else:
            raise ValueError(f"Unknown model name: {name}")

    def fit_tranformers(self, X):
        self.t1 = RandomDilatedShapeletTransform(n_jobs=-1, random_state=self.random_state)
        self.t2 = QUANTTransformer()
        self.t3 = MultiRocket(n_jobs=-1, random_state=self.random_state)
        self.t4 = HydraTransformer(n_jobs=-1, random_state=self.random_state)
        self.t5 = Catch22(n_jobs=-1)
        self.t1.fit(X)
        self.t2.fit(X)
        self.t3.fit(X)
        self.t4.fit(X)
        self.t5.fit(X)

    def transform_series(self, X):
        
        start = perf_counter()
        Xt1 = self.t1.transform(X)
        t1_time = perf_counter() - start
        print(f"RDST transform: {t1_time:.4f}s")
        
        start = perf_counter()
        Xt2 = self.t2.transform(X)
        t2_time = perf_counter() - start
        print(f"QUANT transform: {t2_time:.4f}s")
        
        start = perf_counter()
        Xt3 = self.t3.transform(X)
        t3_time = perf_counter() - start
        print(f"MultiRocket transform: {t3_time:.4f}s")
        
        start = perf_counter()
        Xt4 = self.t4.transform(X)
        t4_time = perf_counter() - start
        print(f"Hydra transform: {t4_time:.4f}s")
        
        start = perf_counter()
        Xt5 = self.t5.transform(X)
        t5_time = perf_counter() - start
        print(f"Catch22 transform: {t5_time:.4f}s")

        return pl.DataFrame(np.hstack([Xt1, Xt2, Xt3, Xt4, Xt5]), schema=
            [f'rdst_{i}' for i in range(Xt1.shape[1])] + 
            [f'quant_{i}' for i in range(Xt2.shape[1])] + 
            [f'multirocket_{i}' for i in range(Xt3.shape[1])] + 
            [f'hydra_{i}' for i in range(Xt4.shape[1])] + 
            [f'catch22_{i}' for i in range(Xt5.shape[1])]
        )

    def _fit(self, X, y):
        if self.cv_splits is None:
            self.cv_splits = generate_folds(X, y, n_splits=self.k_folds, n_repetitions=self.n_repetitions, random_state=self.random_state)
        self.fit_tranformers(X)
        self.Xt_ = self.transform_series(X)
        self.trained_models_ = []

        self.tsc_algos = ['rstsf']
        self.feature_algos = ['multirocket-ridgecv', 'quant-etc', 'rdst-ridgecv']
        self.stacking_models = ['probability-ridgecv', 'probability-et']

        for model_name in self.tsc_algos:
            oof_proba = []
            model_group = []
            for train_idx, val_idx in tqdm(self.cv_splits):
                pipe = self.get_model(model_name)
                pipe.fit(X[train_idx], y[train_idx])
                proba = pipe.predict_proba(X[val_idx])

                prob_columns = []
                for idx, p in zip(val_idx, proba):
                    d = {
                        'index': idx,
                    }
                    for scls, prob in zip(pipe.classes_, p):
                        k = f'proba_model0_{model_name}_class_{scls}'
                        d[k] = prob.item()
                        if k not in prob_columns:
                            prob_columns.append(k)
                    oof_proba.append(d)
                model_group.append(pipe)
            self.trained_models_.append((model_name, model_group))
            agg_probs = pl.DataFrame(oof_proba).group_by('index').mean().sort('index').select(prob_columns)
            self.Xt_ = pl.concat([self.Xt_, agg_probs], how='horizontal')

            # for each model compute oof accuracy
            oof_probas = agg_probs.to_numpy()
            oof_pred_indices = np.argmax(oof_probas, axis=1)
            oof_preds = self.classes_[oof_pred_indices]
            oof_acc = accuracy_score(y, oof_preds)
            print(f"Model {model_name}| CA: {oof_acc:.4f}")

        for model_name in self.feature_algos:
            oof_proba = []
            model_group = []
            for train_idx, val_idx in tqdm(self.cv_splits):
                pipe = self.get_model(model_name)
                pipe.fit(self.Xt_[train_idx], y[train_idx])
                proba = pipe.predict_proba(self.Xt_[val_idx])

                prob_columns = []
                for idx, p in zip(val_idx, proba):
                    d = {
                        'index': idx,
                    }
                    for scls, prob in zip(pipe.classes_, p):
                        k = f'proba_model0_{model_name}_class_{scls}'
                        d[k] = prob.item()
                        if k not in prob_columns:
                            prob_columns.append(k)
                    oof_proba.append(d)
                model_group.append(pipe)
            self.trained_models_.append((model_name, model_group))
            agg_probs = pl.DataFrame(oof_proba).group_by('index').mean().sort('index').select(prob_columns)
            self.Xt_ = pl.concat([self.Xt_, agg_probs], how='horizontal')

            # for each model compute oof accuracy
            oof_probas = agg_probs.to_numpy()
            oof_pred_indices = np.argmax(oof_probas, axis=1)
            oof_preds = self.classes_[oof_pred_indices]
            oof_acc = accuracy_score(y, oof_preds)
            # ocmute also log loss
            #from sklearn.metrics import log_loss, roc_auc_score
            #log_loss_value = log_loss(y, oof_probas)
            # ocmpute AUC 
            #if len(np.unique(y)) == 2:
            #    auc_value = roc_auc_score(y, oof_probas[:, 1])
            #else:
            #    auc_value = roc_auc_score(y, oof_probas, multi_class='ovr', average='macro')
            #print(f"Model {model_name}| CA: {oof_acc:.4f}, Log Loss: {log_loss_value:.4f}, AUC: {auc_value:.4f}")
            print(f"Model {model_name}| CA: {oof_acc:.4f}")

        for model_name in self.stacking_models:
            oof_proba = []
            model_group = []
            for train_idx, val_idx in tqdm(self.cv_splits):
                pipe = self.get_model(model_name)
                pipe.fit(self.Xt_[train_idx], y[train_idx])
                proba = pipe.predict_proba(self.Xt_[val_idx])

                prob_columns = []
                for idx, p in zip(val_idx, proba):
                    d = {
                        'index': idx,
                    }
                    for scls, prob in zip(pipe.classes_, p):
                        k = f'proba_model1_{model_name}_class_{scls}'
                        d[k] = prob.item()
                        if k not in prob_columns:
                            prob_columns.append(k)
                    oof_proba.append(d)
                model_group.append(pipe)
            self.trained_models_.append((model_name, model_group))
            agg_probs = pl.DataFrame(oof_proba).group_by('index').mean().sort('index').select(prob_columns)
            self.Xt_ = pl.concat([self.Xt_, agg_probs], how='horizontal')

            oof_probas = agg_probs.to_numpy()
            oof_pred_indices = np.argmax(oof_probas, axis=1)
            oof_preds = self.classes_[oof_pred_indices]
            oof_acc = accuracy_score(y, oof_preds)

            print(f"Model {model_name}| CA: {oof_acc:.4f}")

        stats_df = []
        for model in self.tsc_algos + self.feature_algos + self.stacking_models:
            stack = '0' if model in self.tsc_algos + self.feature_algos else '1'
            model_columns = [col for col in self.Xt_.columns if col.startswith(f'proba_model{stack}_{model}_class_')]
            #print(model_columns)
            oof_proba = self.Xt_.select(model_columns)
            #print(oof_proba)
            oof_pred_indices = np.argmax(oof_proba, axis=1)
            oof_preds = self.classes_[oof_pred_indices]
            oof_acc = accuracy_score(y, oof_preds)
            #print(f"{model}:{oof_acc:.4f}")
            stats_df.append({
                'model': model,
                'stack': stack,
                'oof_acc': oof_acc,
            })

        stats_df = pl.DataFrame(stats_df).with_columns(
            pl.when(pl.col('model') == 'probability-ridgecv')
            .then(pl.lit(1))
            .otherwise(pl.lit(0))
            .alias('is_preferred')
        ).filter(pl.col('stack')=='1').sort(
            ['oof_acc', 'stack', 'is_preferred'], descending=True
        )
        # print(stats_df)

        self.best_model = (
            stats_df
            .row(0, named=True)
        )['model']
        print(f"Best model selected for prediction: {self.best_model}")



    def predict_proba_per_model(self, X):
        Xt = self.transform_series(X)
        return_dict = {}
        for model_name, model_group in self.trained_models_:
            oof_proba = []
            for model in model_group:
                if model_name in self.tsc_algos:
                    proba = model.predict_proba(X)
                else:
                    proba = model.predict_proba(Xt)
                prob_columns = []
                for i, p in enumerate(proba):
                    d = {
                        'index': i,
                    }
                    for scls, prob in zip(model.classes_, p):
                        model_stack_number = '0' if model_name in self.tsc_algos + self.feature_algos else '1'
                        k = f'proba_model{model_stack_number}_{model_name}_class_{scls}'
                        d[k] = prob.item()
                        if k not in prob_columns:
                            prob_columns.append(k)
                    oof_proba.append(d)
            df = pl.DataFrame(oof_proba).group_by('index').mean().sort('index').select(prob_columns)
            Xt = pl.concat([Xt, df], how='horizontal')
            return_dict[model_name] = df.to_numpy()
        return return_dict


    def _predict_proba(self, X):
        return self.predict_proba_per_model(X)[self.best_model]

    def _predict(self, X):
        probas = self._predict_proba(X)
        predicted_indices = np.argmax(probas, axis=1)
        return self.classes_[predicted_indices]

In [4]:
random_state = 1047
n_repetitions=1
s1 = Stacker(random_state=random_state, n_repetitions=n_repetitions)
s2 = StackerV4(random_state=random_state, n_repetitions=n_repetitions)

In [5]:
s2.fit(X_train, y_train)
pred = s2.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

StratifiedKFold failed, falling back to regular KFold with n_splits=10
RDST transform: 0.3864s
QUANT transform: 0.1068s
MultiRocket transform: 0.0642s
Hydra transform: 0.5349s
Catch22 transform: 0.5005s


100%|██████████| 10/10 [00:09<00:00,  1.05it/s]


Model rstsf| CA: 0.7667


100%|██████████| 10/10 [00:05<00:00,  1.97it/s]


Model multirocket-ridgecv| CA: 0.5667


...

100%|██████████| 10/10 [00:00<00:00, 11.87it/s]


Model probability-ridgecv| CA: 0.5333


100%|██████████| 10/10 [00:05<00:00,  1.75it/s]


Model probability-et| CA: 0.6333
Best model selected for prediction: probability-et
RDST transform: 0.3501s
QUANT transform: 0.0982s
MultiRocket transform: 0.0352s
Hydra transform: 0.5038s
Catch22 transform: 0.0305s


0.8

In [6]:
P = s2.predict_proba_per_model(X_test).items()
print()
for model, prov_pred in P:
    oof_pred_indices = np.argmax(prov_pred, axis=1)
    oof_preds = s2.classes_[oof_pred_indices]
    oof_acc = accuracy_score(y_test, oof_preds)
    print(f"Model {model}| CA: {oof_acc:.4f}")

RDST transform: 0.4215s
QUANT transform: 0.1185s
MultiRocket transform: 0.0366s
Hydra transform: 0.7087s
Catch22 transform: 0.1085s

Model rstsf| CA: 0.8667
Model multirocket-ridgecv| CA: 0.7667
Model quant-etc| CA: 0.7667
Model rdst-ridgecv| CA: 0.8667
Model probability-ridgecv| CA: 0.8000
Model probability-et| CA: 0.8000


In [7]:
s2.Xt_.select([c for c in s2.Xt_.columns if c.startswith('proba_model0')])

proba_model0_rstsf_class_1,proba_model0_rstsf_class_2,proba_model0_rstsf_class_3,proba_model0_rstsf_class_4,proba_model0_rstsf_class_5,proba_model0_multirocket-ridgecv_class_1,proba_model0_multirocket-ridgecv_class_2,proba_model0_multirocket-ridgecv_class_3,proba_model0_multirocket-ridgecv_class_4,proba_model0_multirocket-ridgecv_class_5,proba_model0_quant-etc_class_1,proba_model0_quant-etc_class_2,proba_model0_quant-etc_class_3,proba_model0_quant-etc_class_4,proba_model0_quant-etc_class_5,proba_model0_rdst-ridgecv_class_1,proba_model0_rdst-ridgecv_class_2,proba_model0_rdst-ridgecv_class_3,proba_model0_rdst-ridgecv_class_4,proba_model0_rdst-ridgecv_class_5
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.03,0.14,0.57,0.15,0.11,0.0,0.0,1.0,0.0,0.0,0.065,0.17,0.52,0.09,0.155,0.0,1.0,0.0,0.0,0.0
0.97,0.0,0.01,0.02,0.0,1.0,0.0,0.0,0.0,0.0,0.74,0.05,0.055,0.115,0.04,1.0,0.0,0.0,0.0,0.0
0.81,0.04,0.02,0.07,0.06,1.0,0.0,0.0,0.0,0.0,0.765,0.065,0.05,0.095,0.025,1.0,0.0,0.0,0.0,0.0
0.82,0.02,0.09,0.01,0.06,1.0,0.0,0.0,0.0,0.0,0.63,0.055,0.12,0.04,0.155,1.0,0.0,0.0,0.0,0.0
0.4,0.01,0.04,0.54,0.01,1.0,0.0,0.0,0.0,0.0,0.465,0.055,0.115,0.35,0.015,1.0,0.0,0.0,0.0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0.0,0.15,0.12,0.01,0.72,0.0,0.0,0.0,0.0,1.0,0.005,0.125,0.17,0.01,0.69,0.0,0.0,0.0,0.0,1.0
0.13,0.23,0.25,0.0,0.39,0.0,1.0,0.0,0.0,0.0,0.125,0.255,0.32,0.025,0.275,0.0,1.0,0.0,0.0,0.0
0.02,0.65,0.13,0.07,0.13,0.0,1.0,0.0,0.0,0.0,0.025,0.74,0.06,0.06,0.115,0.0,1.0,0.0,0.0,0.0


In [8]:
for model in s2.tsc_algos:
    proba_cols = [c for c in s2.Xt_.columns if c.startswith(f'proba_model0_{model}')]
    v = s2.Xt_.select(proba_cols).to_numpy()

    oof_pred_indices = np.argmax(v, axis=1)
    oof_preds = s2.classes_[oof_pred_indices]
    oof_acc = accuracy_score(y_test, oof_preds)
    print(f"Model: {model}", oof_acc)

Model: rstsf 0.7666666666666667


In [ ]:
class MeanTopN:
    def __init__(self, n=2):
        self.n = n

    def fit(self, X, y):
        pass

    def predict_proba(self, probas):
        pass

class MeanMaxTreshold:
    def __init__(self, threshold=0.95):
        self.threshold = threshold

    def fit(self, X, y):
        pass

    def predict_proba(self, probas):
        pass

array([[0.03, 0.14, 0.57, 0.15, 0.11],
       [0.97, 0.  , 0.01, 0.02, 0.  ],
       [0.81, 0.04, 0.02, 0.07, 0.06],
       [0.82, 0.02, 0.09, 0.01, 0.06],
       [0.4 , 0.01, 0.04, 0.54, 0.01],
       [0.11, 0.2 , 0.5 , 0.06, 0.13],
       [0.01, 0.52, 0.11, 0.04, 0.32],
       [0.  , 0.54, 0.18, 0.03, 0.25],
       [0.05, 0.5 , 0.12, 0.05, 0.28],
       [0.  , 0.07, 0.23, 0.16, 0.54],
       [0.02, 0.52, 0.27, 0.01, 0.18],
       [0.  , 0.43, 0.2 , 0.01, 0.36],
       [0.09, 0.1 , 0.47, 0.09, 0.25],
       [0.  , 0.09, 0.59, 0.03, 0.29],
       [0.02, 0.32, 0.37, 0.06, 0.23],
       [0.07, 0.11, 0.43, 0.32, 0.07],
       [0.05, 0.43, 0.29, 0.09, 0.14],
       [0.  , 0.02, 0.59, 0.04, 0.35],
       [0.02, 0.08, 0.19, 0.59, 0.12],
       [0.  , 0.31, 0.28, 0.33, 0.08],
       [0.03, 0.09, 0.14, 0.73, 0.01],
       [0.03, 0.07, 0.11, 0.72, 0.07],
       [0.  , 0.01, 0.08, 0.89, 0.02],
       [0.03, 0.01, 0.05, 0.91, 0.  ],
       [0.02, 0.34, 0.17, 0.01, 0.46],
       [0.  , 0.15, 0.12,

In [10]:
sdfsdf0sdsdg=srgrsgrtg

NameError: name 'srgrsgrtg' is not defined

In [ ]:
m = MultiRocketHydraClassifier(n_jobs=-1, random_state=random_state)
m.fit(X_train, y_train)
pred = m.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

In [ ]:
from aeon.classification.interval_based import RSTSF
m = RSTSF(random_state=random_state, n_jobs=-1)
m.fit(X_train, y_train)
pred = m.predict(X_test)    
acc = accuracy_score(y_test, pred)
acc

In [ ]:
from aeon.classification.hybrid import HIVECOTEV2
m = HIVECOTEV2(time_limit_in_minutes=3, random_state=random_state, n_jobs=-1)
m.fit(X_train, y_train)
pred = m.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

In [ ]:
from aeon.classification.interval_based import DrCIFClassifier
m = DrCIFClassifier(random_state=random_state, n_jobs=-1)
m.fit(X_train, y_train)
pred = m.predict(X_test)    
acc = accuracy_score(y_test, pred)
acc

In [ ]:
from aeon.classification.dictionary_based import TemporalDictionaryEnsemble
m = TemporalDictionaryEnsemble(random_state=random_state, n_jobs=-1)
m.fit(X_train, y_train)
pred = m.predict(X_test)    
acc = accuracy_score(y_test, pred)
acc

In [ ]:
from aeon.classification.shapelet_based import ShapeletTransformClassifier
m = ShapeletTransformClassifier(random_state=random_state, n_jobs=-1)
m.fit(X_train, y_train)
pred = m.predict(X_test)    
acc = accuracy_score(y_test, pred)
acc

In [ ]:
X_train.shape

In [ ]:
from aeon.transformations.collection.shapelet_based import SAST
from aeon.datasets import load_unit_test

sast = SAST(n_jobs=-1)
sast.fit(X_train, y_train)

sast.transform(X_train)

In [ ]:
fsdf=dfgdgf

In [ ]:
s1.fit(X_train, y_train)
pred = s1.predict(X_test)
acc = accuracy_score(y_test, pred)
acc

In [ ]:
import numpy as np
from sklearn.linear_model import RidgeClassifierCV



# Generate random data
np.random.seed(42)
X_train = np.random.randn(100, 20)  # 100 samples, 20 features
y_train = np.random.randint(0, 3, 100)  # 3 classes

X_test = np.random.randn(30, 20)
y_test = np.random.randint(0, 3, 30)

# Train RidgeClassifierCV
model = RidgeClassifierCVIndicator(alphas=[0.1, 1.0, 10.0, 100.0])
model.fit(X_train, y_train)

# Predict
predictions = model.predict(X_test)
score = model.score(X_test, y_test)

print(f"Best alpha: {model.alpha_}")
print(f"Test accuracy: {score:.3f}")
print(f"Predictions: {predictions[:10]}")

In [ ]:
model.predict_proba(X_test)

In [ ]:
model.predict(X_test)